In [53]:
"""This script generates the features for classification in subtask A. Set the TRAIN flag to True to 
    generate features for the training data, and to False to generate features for test data"""

import json, re
import numpy as np 
import pandas as pd

TRAIN = False

In [3]:
import os
print("Current working directory:", os.getcwd())

import torch
print("Torch version:", torch.__version__)

from transformers.utils import is_torch_available
print("Transformers sees torch:", is_torch_available())

Current working directory: d:\FERI - MAG\1. letnik\2. semester\JT\irony_detection\slovene_pipeline\subtaskA_notebooks
Torch version: 2.1.2+cpu


c:\Users\jer19\anaconda3\envs\jt\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers sees torch: True


In [ ]:
# Download required NLTK data
import nltk
import ssl

# Handle SSL certificate issues
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download all required resources
# Note: newer NLTK versions renamed averaged_perceptron_tagger to averaged_perceptron_tagger_eng
resources = [
    'wordnet',
    'averaged_perceptron_tagger',
    'averaged_perceptron_tagger_eng',
    'sentiwordnet',
    'omw-1.4'
]
for resource in resources:
    try:
        nltk.download(resource)
    except Exception as e:
        print(f"Failed to download {resource}: {e}")

print("âœ“ NLTK data download complete!")

In [45]:
# reading the data
from load import parse_dataset

In [54]:
if TRAIN:
    dataset='../../datasets/slovene/train/SemEval2018-T3-train-taskA_emoji.txt'
    corpus, _ = parse_dataset(dataset)
    corpus_preprocessed = json.load(open('../../extra_resources/train_preprocessed.txt','r'))
else:
    dataset='../../datasets/slovene/test_TaskA/SemEval2018-T3_input_test_taskA_emoji.txt'
    corpus = parse_dataset(dataset)
    corpus_preprocessed = json.load(open('../../extra_resources/test_preprocessed.txt','r'))

In [55]:
# intensity features - 3 binarized features for splitted tweets which show:
# 1) the intensity of the left half
# 2) the intensity of the right half
# 3) the difference between the polarities of left and right halves

from functools import lru_cache

# Use a multilingual sentiment model (works for Slovene) instead of rule-only heuristics.
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"


def chunkIt(seq, n):
    """splits the list into n approximately equal sub-lists. source: goo.gl/VrHKeR"""
    avg = len(seq) / float(n)
    out = []
    last = 0.0
    while last < len(seq):
        out.append(seq[int(last):int(last + avg)])
        last += avg
    return out


@lru_cache(maxsize=1)
def get_sentiment_pipeline():
    """Lazy-load multilingual sentiment pipeline once per session."""
    try:
        from transformers import pipeline
        return pipeline("sentiment-analysis", model=MODEL_NAME, tokenizer=MODEL_NAME)
    except Exception as e:
        print(f"Warning: could not load sentiment model ({e}). Falling back to lexicon scoring.")
        return None


sentiment_cache = {}


def _fallback_sentiment(text_lower):
    """Fallback heuristic if model is unavailable."""
    positive_words = {
        'rad', 'obožujem', 'lepo', 'super', 'odlično', 'hvala', 'fantastično',
        'izvrstno', 'odličen', 'love', 'perfect', 'great'
    }
    negative_words = {
        'zaprta', 'zaprto', 'zaprte', 'slabo', 'čudno', 'groza', 'strašno',
        'problematično', 'grozno', 'hudo', 'hate', 'terrible', 'awful', 'bad'
    }

    words = text_lower.split()
    pos_count = sum(1 for w in words if any(w.startswith(p) for p in positive_words))
    neg_count = sum(1 for w in words if any(w.startswith(n) for n in negative_words))

    if pos_count > neg_count:
        return 3
    if neg_count > pos_count:
        return 1
    return 2


def mock_sentiment_score(text):
    """Model-based sentiment score mapped to the original 0..4 scale."""
    if isinstance(text, list):
        text = ' '.join(text)

    text = (text or '').strip()
    if not text:
        return 2  # neutral

    cache_key = text.lower()
    if cache_key in sentiment_cache:
        return sentiment_cache[cache_key]

    nlp = get_sentiment_pipeline()
    if nlp is None:
        score = _fallback_sentiment(cache_key)
        sentiment_cache[cache_key] = score
        return score

    try:
        pred = nlp(text[:512])[0]
        label = str(pred.get('label', '')).lower()
        conf = float(pred.get('score', 0.0))

        if label in {'label_0', 'negative'}:
            score = 0 if conf >= 0.85 else 1
        elif label in {'label_1', 'neutral'}:
            score = 2
        else:  # label_2 / positive
            score = 4 if conf >= 0.85 else 3
    except Exception:
        score = _fallback_sentiment(cache_key)

    sentiment_cache[cache_key] = score
    return score


feats_1 = []

for text in corpus:
    part1, part2 = chunkIt(text, 2)
    output1 = mock_sentiment_score(part1)
    output2 = mock_sentiment_score(part2)

    leftIntensity = rightIntensity = polarityDiff = 0
    if output1 in [0, 4]:
        leftIntensity = 1
    if output2 in [0, 4]:
        rightIntensity = 1
    if (output1 > 2 and output2 < 2) or (output1 < 2 and output2 > 2):
        polarityDiff = 1
    feats_1.append(np.array([leftIntensity, rightIntensity, polarityDiff]))

In [56]:
# contrast
# Uses multilingual sentiment model from the previous cell for text/hashtag sentiment.
df = pd.read_csv('../../extra_resources/Emoji_Sentiment_Data_v1.0.csv')

df = df[['Emoji', 'Negative', 'Neutral', 'Positive']]
tuples = [tuple(x) for x in df.values]
# tuples are of the form (emoji, negative, neutral, positive)

idx2lb = {0: -1, 1: 0, 2: 1}
emoji_sentimens = {}
for val in tuples:
    emoji_sentimens[val[0]] = idx2lb[np.argmax(np.array(val[1:]))]


def extractEmoticon(tweet):
    """returns all the emoticons in tweet"""
    return re.findall(r'[\U0001f600-\U0001f650]', ' '.join(tweet))


twts = [extractEmoticon(twt[0]) for twt in corpus_preprocessed]
twts = [[emoji_sentimens[emoji] for emoji in twt if emoji in emoji_sentimens] for twt in twts]


def extractHashtag(tweet):
    t = tweet.split(' ')
    text = []
    hashtagText = []
    oneHashtag = []
    flag = 0
    for w in t:
        if w == "<hashtag>":
            flag = 1
            continue
        if flag == 1:
            if w == "</hashtag>":
                hashtagText.append(oneHashtag)
                oneHashtag = []
                flag = 0
            else:
                oneHashtag.append(w)
        else:
            text.append(w)
    return text, hashtagText


txt = [extractHashtag(tweet) for tweet in corpus_preprocessed]

assert len(txt) == len(twts)
txt = [(txt[i][0], txt[i][1], twts[i]) for i in range(len(twts))]

# 0: very negative
# 1: negative
# 2: neutral
# 3: positive
# 4: very positive

def sentiment(txt):
    """Compute sentiment with the model-based scorer from the previous cell."""
    txt_str = ' '.join(txt) if isinstance(txt, list) else txt
    if not len(txt_str):
        return 2
    return mock_sentiment_score(txt_str)


def contrast(twt):
    """search for emotion contrast in hashtag, emoticon and tweet text"""
    contrast = 0
    txt_sentiment = sentiment(twt[0])
    htag_sentiment = [sentiment(hash_segment) for hash_segment in twt[1]]
    emoji_sentiment = twt[2]

    hset = set(htag_sentiment)
    eset = set(emoji_sentiment)

    if (txt_sentiment in {2, 3, 4}) and (hset & {0, 1}):
        contrast = 1
    elif (txt_sentiment in {0, 1}) and (hset & {3, 4}):
        contrast = 1
    elif (txt_sentiment in {2, 3, 4}) and (-1 in eset):
        contrast = 1
    elif (txt_sentiment in {0, 1}) and (1 in eset):
        contrast = 1
    elif {-1, 1}.issubset(eset):
        contrast = 1
    elif {0, 4}.issubset(hset) or {0, 3}.issubset(hset) or {1, 4}.issubset(hset):
        contrast = 1
    elif (hset & {0, 1}) and (1 in eset):
        contrast = 1
    elif (hset & {3, 4}) and (-1 in eset):
        contrast = 1
    return contrast


contrast_feats = [np.array([contrast(twt)]) for twt in txt]

In [ ]:
# feats = [feats[i].append(contrast_feats[i]) for i in range(len(feats))]

In [57]:
# ekphrasis-based features (extracted from pre-processed data)

tags =  ['<allcaps>', '<annoyed>', '<censored>', '<date>', '<elongated>', '<emphasis>', '<happy>',
         '<hashtag>', '<heart>', '<kiss>', '<laugh>', '<money>', '<number>', '<percent>', '<phone>',
         '<repeated>', '<sad>', '<shocking>', '<surprise>', '<time>', '<tong>', '<url>', '<user>',
         '<wink>']

def tweet_vecs(twt, n=2):
    """extract a feature vector for a single tweet, based on the counts of the annotation tags
        split the tweet to n equal parts and computes the same features for each part"""
    twt = twt.split()
    chunks = chunkIt(twt, n)
    
    scores = []
    
    for chunk in chunks:
        for tag in tags:
            scores.append(sum(1 for t in chunk if t == tag))
    return scores
    
def feats(text):
    """apply the tweet_vecs function on all tweets and return a result in a list"""
    return [tweet_vecs(twt) for twt in text]

ekphrasis_feats = [np.array(v) for v in feats(corpus_preprocessed)]

In [58]:
from ekphrasis.utils.nlp import polarity
polarity_flag = True

polarity_vectors = []
for tweet in corpus_preprocessed:
    chunks = chunkIt(tweet, 2)
    polarity_vectors.append(np.concatenate(((polarity(chunks[0])[1], polarity(chunks[1])[1])), axis=0))

assert len(ekphrasis_feats) == len(polarity_vectors)

if polarity_flag: 
    ekphrasis_feats = [np.concatenate((ekphrasis_feats[i], polarity_vectors[i])) for i in range(len(ekphrasis_feats))]

In [ ]:
ekphrasis_feats[0].shape

In [59]:
# Debug: check the sizes of different feature arrays
print(f"feats_1 size: {len(feats_1)}")
print(f"contrast_feats size: {len(contrast_feats)}")
print(f"ekphrasis_feats size: {len(ekphrasis_feats)}")
print(f"corpus size: {len(corpus)}")
print(f"corpus_preprocessed size: {len(corpus_preprocessed)}")
print(f"txt size: {len(txt)}")
print(f"twts size: {len(twts)}")

# Ensure all arrays have the same length
# Use the minimum length across all arrays
min_len = min(len(feats_1), len(contrast_feats), len(ekphrasis_feats))
print(f"\nUsing min length: {min_len}")

# Truncate all arrays to the same length
feats_1 = feats_1[:min_len]
contrast_feats = contrast_feats[:min_len]
ekphrasis_feats = ekphrasis_feats[:min_len]

# Base feature matrix: 58 dims
features = np.concatenate((feats_1, contrast_feats, ekphrasis_feats), axis=1)

# Keep explicit hashtag/emoji sentiment priors in the last two base slots (58-dim schema).
# feature[56]: emoji sentiment prior in [0,1], higher means stronger negative/ironic emoji signal
# feature[57]: hashtag sentiment-contrast prior in [0,1], higher means mixed/opposing hashtag sentiment

def _emoji_sent_prior(emoji_sent_list):
    if not emoji_sent_list:
        return 0.0
    neg_ratio = sum(1 for v in emoji_sent_list if v < 0) / len(emoji_sent_list)
    mixed = 1.0 if ({-1, 1}.issubset(set(emoji_sent_list))) else 0.0
    return min(1.0, 0.7 * neg_ratio + 0.3 * mixed)


def _hashtag_sent_prior(hashtags):
    if not hashtags:
        return 0.0

    # Each hashtag segment is a list of tokens.
    seg_scores = [sentiment(seg) for seg in hashtags if len(seg) > 0]
    if not seg_scores:
        return 0.0

    has_pos = any(s >= 3 for s in seg_scores)
    has_neg = any(s <= 1 for s in seg_scores)
    mixed = 1.0 if (has_pos and has_neg) else 0.0
    polarity_spread = (max(seg_scores) - min(seg_scores)) / 4.0
    return min(1.0, 0.6 * mixed + 0.4 * polarity_spread)


emoji_sentiment_priors = []
hashtag_sentiment_priors = []

for i in range(min_len):
    # txt[i] structure: (tweet_text_tokens, hashtag_segments, emoji_sentiment_values)
    hashtag_segments = txt[i][1]
    emoji_sent_vals = txt[i][2]

    emoji_sentiment_priors.append(_emoji_sent_prior(emoji_sent_vals))
    hashtag_sentiment_priors.append(_hashtag_sent_prior(hashtag_segments))

features[:, 56] = np.array(emoji_sentiment_priors)
features[:, 57] = np.array(hashtag_sentiment_priors)

print(f"Injected emoji sentiment prior into feature 56 (mean={np.mean(features[:, 56]):.4f})")
print(f"Injected hashtag sentiment prior into feature 57 (mean={np.mean(features[:, 57]):.4f})")

# Add pragmatic priors as two NEW columns -> total 60 dims.
# Make these soft and capped so they assist rather than dominate.
EMOJI_IRONY_SET = {'🙄', '😒', '😑', '🤦', '🤦\u200d♂️', '🤦\u200d♀️', '😬', '😏', '🙃'}
SLO_INCONV_PATTERNS = [
    r'\b(še\s+dobro\s+da)\b',
    r'\b(super|odlično|fajn|hvala|vrhunsko)\b.*\b(ko|da)\b.*\b(zamujam|dežuje|zaprta|gužva|gneča|pokvar|crkn|čakanja|čakalnici)'
]
NEG_EVENT_HINTS = ['zamujam', 'dežuje', 'zaprta', 'gužva', 'gneča', 'pokvar', 'crkn', 'čakanja', 'čakalnici']
POS_IRONIC_OPENERS = ['super', 'odlično', 'fajn', 'hvala', 'vrhunsko']


def _to_text(val):
    if isinstance(val, (list, tuple)):
        return ' '.join(str(x) for x in val)
    return str(val)


def _emoji_irony_score(text):
    hits = sum(text.count(e) for e in EMOJI_IRONY_SET)
    # capped in [0, 0.35]
    return min(0.35, 0.12 * min(hits, 2) + (0.08 if hits > 0 else 0.0))


def _inconvenience_score(text_lower):
    pat_hits = sum(1 for pat in SLO_INCONV_PATTERNS if re.search(pat, text_lower))
    neg_hits = sum(1 for tok in NEG_EVENT_HINTS if tok in text_lower)
    opener = 1 if any(tok in text_lower for tok in POS_IRONIC_OPENERS) else 0
    # capped in [0, 0.35]
    return min(0.35, 0.10 * pat_hits + 0.05 * min(neg_hits, 3) + 0.05 * opener)


emoji_irony_scores = []
inconvenience_scores = []
for tw in corpus[:min_len]:
    tw_text = _to_text(tw)
    tw_low = tw_text.lower()

    emoji_irony_scores.append(_emoji_irony_score(tw_text))
    inconvenience_scores.append(_inconvenience_score(tw_low))

pragmatic_extra = np.column_stack((np.array(emoji_irony_scores), np.array(inconvenience_scores)))
features = np.concatenate((features, pragmatic_extra), axis=1)

print(f"Added EMOJI_IRONY_SET soft prior as feature 58 (mean={np.mean(features[:, 58]):.4f})")
print(f"Added SLO_INCONV_PATTERNS soft prior as feature 59 (mean={np.mean(features[:, 59]):.4f})")
print(f"Final feature matrix shape: {features.shape}")

feats_1 size: 641
contrast_feats size: 784
ekphrasis_feats size: 784
corpus size: 641
corpus_preprocessed size: 784
txt size: 784
twts size: 784

Using min length: 641
Injected emoji sentiment prior into feature 56 (mean=0.0000)
Injected hashtag sentiment prior into feature 57 (mean=0.0348)
Added EMOJI_IRONY_SET soft prior as feature 58 (mean=0.0087)
Added SLO_INCONV_PATTERNS soft prior as feature 59 (mean=0.0087)
Final feature matrix shape: (641, 60)


In [ ]:
len(features)

In [ ]:
features.shape

In [60]:
# save the features in a numpy file 
if TRAIN:
    np.save('train_feats_taskA.npy', features)
else:
    np.save('test_feats_taskA.npy', features)